# STEP 1. 원본 적재 + 스키마 통일

**H2**: 골목상권의 순감소 위험이 높은 업종의 경우, 점포 규모와 유동인구를 맞추면 타 업종과 순감소 격차가 줄어들 것이다.

## 이 노트북에서 쓰는 데이터

| 파일 | 역할 |
|---|---|
| 점포-상권 2023 / 2024 / 2025~ | 라벨(개업·폐업) + 처리변수(업종) + 규모 공변량(점포수) |
| 추정매출-상권 2023/2024/2025/_ | 보조 변수 (H2 매칭에는 미사용) |
| 영역-상권 | 상권유형 · 자치구 · 면적 · 좌표 |
| 길단위인구-상권 | **매칭 핵심 공변량** (유동인구) |
| 집객시설-상권 | 매칭 보조 공변량 (입지) |

## 핵심 처리 두 가지
1. **점포 파일 컬럼명 통일** — 2023·2024는 `점포_수`/`유사_업종_점포_수`, 2025~는 `전체_점포_수`/`일반_점포_수`
2. **매출 파일 중복 제거** — 2025년 파일과 `_.csv`가 20251~20254 구간에서 겹침

> `data/raw/` 에 CSV를 넣고 실행하세요.

In [1]:
import sys, os
from pathlib import Path

# 노트북이 어디서 열리든 프로젝트 루트를 찾아 sys.path에 추가
ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
print("프로젝트 루트:", ROOT)

프로젝트 루트: c:\Users\spide\ai-data-bootcamp\project\h2_nb


In [2]:
from config import RAW, PROC, ENCODING, FILES

def read(key: str) -> pd.DataFrame:
    path = RAW / FILES[key]
    if not path.exists():
        raise FileNotFoundError(f"파일 없음: {path}\n  → data/raw/ 에 CSV를 넣어주세요.")
    return pd.read_csv(path, encoding=ENCODING)

# 원본 파일 존재 확인
for k, v in FILES.items():
    mark = "OK" if (RAW / v).exists() else "없음"
    print(f"  [{mark:>3}] {k:12} {v}")

  [ OK] store_2023   서울시_상권분석서비스(점포-상권)_2023년.csv
  [ OK] store_2024   서울시 상권분석서비스(점포-상권)_2024년.csv
  [ OK] store_2025   서울시 상권분석서비스(점포-상권).csv
  [ OK] sales_2023   서울시_상권분석서비스(추정매출-상권)_2023년.csv
  [ OK] sales_2024   서울시 상권분석서비스(추정매출-상권)_2024년.csv
  [ OK] sales_2025   서울시 상권분석서비스(추정매출-상권)_2025년.csv
  [ OK] sales_late   서울시 상권분석서비스(추정매출-상권).csv
  [ OK] area         서울시 상권분석서비스(영역-상권).csv
  [ OK] flow         서울시 상권분석서비스(길단위인구-상권).csv
  [ OK] facility     서울시 상권분석서비스(집객시설-상권).csv


## 1-1. 점포 데이터 — 스키마 통일

두 스키마의 공통 의미 컬럼만 사용합니다.

- `점포_수` ≡ `전체_점포_수` (검증: 일반+프랜차이즈 = 전체, 일치율 100%)
- `유사_업종_점포_수` → 2025~ 파일에 없음 → **폐기**
- `일반_점포_수` → 2023·2024 파일에 없음 → **폐기**

In [3]:
STORE_COMMON = ["기준_년분기_코드", "상권_구분_코드_명", "상권_코드", "상권_코드_명",
                "서비스_업종_코드", "서비스_업종_코드_명", "점포_수",
                "프랜차이즈_점포_수", "개업_율", "개업_점포_수", "폐업_률", "폐업_점포_수"]

frames = []
for key in ["store_2023", "store_2024", "store_2025"]:
    df = read(key)
    if "전체_점포_수" in df.columns:            # 신 스키마 → 구 스키마 이름으로 통일
        df = df.rename(columns={"전체_점포_수": "점포_수"})
    missing = set(STORE_COMMON) - set(df.columns)
    assert not missing, f"{key}: 컬럼 누락 {missing}"
    df = df[STORE_COMMON].copy()
    df["_source"] = key
    frames.append(df)
    print(f"  {key:12} {len(df):>8,}행  분기 {sorted(df['기준_년분기_코드'].unique())}")

store = pd.concat(frames, ignore_index=True)

keys = ["기준_년분기_코드", "상권_코드", "서비스_업종_코드"]
dup = store.duplicated(keys).sum()
print(f"\n  결합 {len(store):,}행 | 키 중복 {dup:,}건")
if dup:
    store = store.drop_duplicates(keys, keep="last")
    print(f"  중복 제거 후 {len(store):,}행")
store.head(3)

  store_2023    307,741행  분기 [np.int64(20231), np.int64(20232), np.int64(20233), np.int64(20234)]
  store_2024    306,889행  분기 [np.int64(20241), np.int64(20242), np.int64(20243), np.int64(20244)]
  store_2025    380,747행  분기 [np.int64(20251), np.int64(20252), np.int64(20253), np.int64(20254), np.int64(20261)]

  결합 995,377행 | 키 중복 0건


,기준_년분기_코드,상권_구분_코드_명,상권_코드,상권_코드_명,서비스_업종_코드,서비스_업종_코드_명,점포_수,프랜차이즈_점포_수,개업_율,개업_점포_수,폐업_률,폐업_점포_수,_source
0,20231,골목상권,3110001,이북5도청사,CS100001,한식음식점,10,1,9,1,0,0,store_2023
1,20231,골목상권,3110001,이북5도청사,CS100003,일식음식점,1,0,0,0,0,0,store_2023
2,20231,골목상권,3110001,이북5도청사,CS100008,분식전문점,3,0,0,0,0,0,store_2023


## 1-2. 매출 데이터 — 중복 제거

`추정매출-상권_.csv` 는 20251~20261을 담고 있어 2025년 파일과 4개 분기가 겹칩니다.

In [4]:
SALES_COLS = ["기준_년분기_코드", "상권_코드", "서비스_업종_코드", "서비스_업종_코드_명",
              "당월_매출_금액", "당월_매출_건수"]

frames = []
for key in ["sales_2023", "sales_2024", "sales_2025", "sales_late"]:
    df = read(key)[SALES_COLS].copy()
    df["_source"] = key
    frames.append(df)
    print(f"  {key:12} {len(df):>8,}행  분기 {sorted(df['기준_년분기_코드'].unique())}")

sales = pd.concat(frames, ignore_index=True)
dup = sales.duplicated(keys).sum()
print(f"\n  결합 {len(sales):,}행 | 키 중복 {dup:,}건  ← 2025년 파일 ∩ _.csv")
sales = sales.drop_duplicates(keys, keep="last")
print(f"  중복 제거 후 {len(sales):,}행")

  sales_2023     88,246행  분기 [np.int64(20231), np.int64(20232), np.int64(20233), np.int64(20234)]
  sales_2024     87,179행  분기 [np.int64(20241), np.int64(20242), np.int64(20243), np.int64(20244)]
  sales_2025     85,732행  분기 [np.int64(20251), np.int64(20252), np.int64(20253), np.int64(20254)]
  sales_late    106,920행  분기 [np.int64(20251), np.int64(20252), np.int64(20253), np.int64(20254), np.int64(20261)]

  결합 368,077행 | 키 중복 85,732건  ← 2025년 파일 ∩ _.csv
  중복 제거 후 282,345행


## 1-3. 단일 파일 3종

In [5]:
area = read("area")
flow = read("flow")
facility = read("facility")

for nm, df, note in [("영역-상권", area, "상권유형·자치구·면적·좌표"),
                     ("길단위인구-상권", flow, "매칭 핵심 공변량"),
                     ("집객시설-상권", facility, "매칭 보조 공변량")]:
    q = sorted(df["기준_년분기_코드"].unique()) if "기준_년분기_코드" in df else ["-"]
    print(f"  {nm:16} {len(df):>7,}행  상권 {df['상권_코드'].nunique():>5}개  "
          f"분기 {q[0]}~{q[-1]} ({len(q)})  · {note}")

  영역-상권              1,650행  상권  1650개  분기 -~- (1)  · 상권유형·자치구·면적·좌표
  길단위인구-상권          34,633행  상권  1650개  분기 20211~20261 (21)  · 매칭 핵심 공변량
  집객시설-상권           33,138행  상권  1578개  분기 20211~20261 (21)  · 매칭 보조 공변량


## 1-4. 상권 마스터 점검

In [6]:
print(area["상권_구분_코드_명"].value_counts().to_string())
print(f"\n영역_면적(㎡) 최소 {area['영역_면적'].min():,.0f} / "
      f"중앙 {area['영역_면적'].median():,.0f} / 최대 {area['영역_면적'].max():,.0f}")
print(f"→ 최대/최소 {area['영역_면적'].max()/area['영역_면적'].min():,.0f}배 "
      f"→ 밀도 계산 시 로그 변환 필수")
print(f"좌표계 EPSG:5181 (미터)  X {area['엑스좌표_값'].min():,.0f}~{area['엑스좌표_값'].max():,.0f}")

상권_구분_코드_명
골목상권    1090
전통시장     305
발달상권     249
관광특구       6

영역_면적(㎡) 최소 1,854 / 중앙 71,928 / 최대 2,462,734
→ 최대/최소 1,328배 → 밀도 계산 시 로그 변환 필수
좌표계 EPSG:5181 (미터)  X 182,509~215,352


In [7]:
store.to_pickle(PROC / "01_store.pkl")
sales.to_pickle(PROC / "01_sales.pkl")
area.to_pickle(PROC / "01_area.pkl")
flow.to_pickle(PROC / "01_flow.pkl")
facility.to_pickle(PROC / "01_facility.pkl")
print(f"[저장] {PROC} 에 01_*.pkl 5개")

[저장] C:\Users\spide\ai-data-bootcamp\project\h2_nb\data\processed 에 01_*.pkl 5개
